### **1. Introduction & Recap**

*   **Problem:** Large Language Models (LLMs) naturally return **unstructured textual responses**. This makes it difficult to integrate them directly with other systems like databases or APIs that require structured data.
*   **Recap from Previous Video (Structured Output):**
    *   **Goal:** Force the LLM to return a structured response (like JSON) instead of plain text.
    *   **Two Types of LLMs:**
        1.  **Models that can natively give structured output** (e.g., newer GPT models). In LangChain, you can use the `with_structured_output` method for these.
        2.  **Models that cannot natively give structured output** (e.g., many open-source models like TinyLlama). These require manual guidance.
*   **Solution for Today's Video:** **Output Parsers**. They help convert the raw, unstructured text from *any* LLM into a desired structured format.

### **2. What are Output Parsers?**

*   **Definition:** Classes in LangChain that help convert raw LLM responses (text) into structured formats like **JSON, CSV, Pydantic models**, etc.
*   **Purpose:** They ensure **consistency**, **validation**, and **ease of use** in your applications.
*   **Key Point:** They work seamlessly with **both** types of LLMs (those that can and cannot natively produce structured output).
*   **Four Main Parsers Covered:**
    1.  String Output Parser
    2.  JSON Output Parser
    3.  Structured Output Parser
    4.  Pydantic Output Parser

---

### **3. Parser 1: String Output Parser**

*   **Purpose:** The simplest parser. It extracts just the **textual content** from an LLM response, stripping away any metadata (like token usage, etc.).
*   **When to Use:** When you need the plain text response and want to use it in a processing pipeline (chain).
*   **Analogy:** It's like always doing `result.content` automatically.

### 1. Initialize Model and Parser

In [ ]:

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

api_key = os.getenv("Hugging_face_api_token")

# Create LLM endpoint
llm = HuggingFaceEndpoint(
    # repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    huggingfacehub_api_token=api_key,
    temperature=0.5,
)

# Wrap with chat interface
model = ChatHuggingFace(llm=llm)


In [ ]:
# --- Imports ---
from langchain_core.output_parsers import StrOutputParser

string_parser=StrOutputParser()

### 2. Create Prompt Templates 

In [5]:
from langchain_core.prompts import PromptTemplate
# 1st prompt->Detailed report
template1 = PromptTemplate(
    template="Write a detailed report on {topic}", input_variables=["topic"]
)
# 2nd prompt->summary
template2 = PromptTemplate(
    template="Write a 5 line summary on the following text ./n {text}",
    input_variables=["text"],
)


### 3. Create a Sequential Chain (Pipeline)

In [ ]:
# The output of the first step is automatically parsed (as string) and passed to the next.
chain = template1 | model | string_parser | template2| model | string_parser

### 4. Invoke the Chain

In [8]:
result=chain.invoke({
    "topic":"Black Holes" # only input for template1
})

### Output (Final Summary)

In [9]:
result

' I. Black Holes: Cosmic Monsters Revealed - Summary\n\n1. Black holes are celestial bodies formed when massive stars reach the end of their life cycle, collapsing under their own gravity to create an infinitely dense singularity and an event horizon.\n2. Characterized by their mass, spin, and electric charge, black holes are known for their immense gravitational pull, described by their Schwarzschild radius, and the absence of light or electromagnetic radiation emission.\n3. Black holes can interact with nearby matter, causing stars to orbit around them and heating up gas in their vicinity to emit X-rays. When two black holes merge, they create gravitational waves that ripple through space-time.\n4. Latest discoveries in the field include the detection of gravitational waves from black hole mergers by LIGO and Virgo collaborations in 2015, and the observation of a black hole at the center of the Milky Way.\n5. Ongoing research focuses on understanding black hole properties, behavior, 